# Bracket Matching Exhaustive Causal Head Search

Exhaustive zero-ablation causal sweep across all 384 attention heads (24 layers × 16 heads) in SantaCoder on bracket-matching minimal-pair prompts.

**Causal signal:** `ablation_logit_drop = baseline_logit_diff - ablated_logit_diff`, where `logit_diff` is the logit of the correct closer minus the logit of the most probable incorrect closer.

Final output: `bracket_exhaustive_results.csv`.

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformer_lens==2.17.0",
    "transformers==4.57.6",
    "einops==0.8.2",
    "typeguard==4.5.1",
], check=True)

# Patch SantaCoder compatibility
import os, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

_REPO_ID = "bigcode/santacoder"
_PATCHES = {
    "modeling_gpt2_mq.py": [
        (
            "from transformers.modeling_utils import PreTrainedModel, SequenceSummary",
            "from transformers.modeling_utils import PreTrainedModel",
        ),
    ],
}

def patch_file(path, replacements):
    content = path.read_text()
    for target, replacement in replacements:
        content = content.replace(target, replacement)
    path.write_text(content)
    print(f"patched: {path.name}")

for filename, replacements in _PATCHES.items():
    snapshot_path = Path(hf_hub_download(_REPO_ID, filename))
    patch_file(snapshot_path, replacements)

dyn_cache = Path.home() / ".cache" / "huggingface" / "modules" / "transformers_modules" / "bigcode"
if dyn_cache.exists():
    shutil.rmtree(dyn_cache)
    print(f"cleared dynamic cache: {dyn_cache}")

print("Setup complete. Now go to Runtime → Restart session, then run all cells below this one.")

In [ ]:
import torch
from transformer_lens import HookedTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

model = HookedTransformer.from_pretrained(
    "santacoder",
    device=DEVICE,
)
model.cfg.use_attn_result = True
model.eval()

N_LAYERS = model.cfg.n_layers  # 24
N_HEADS  = model.cfg.n_heads   # 16
print(f"n_layers={N_LAYERS}, n_heads={N_HEADS}")

In [ ]:
from dataclasses import dataclass

@dataclass
class BracketPair:
    matched_tokens: list    # prompt ending with the correct closer
    mismatched_tokens: list # identical prompt but wrong closer at the end
    correct_closer_id: int
    incorrect_closer_id: int
    label: str
    closer_type: str        # one of ")", "]", "}"
    depth: int              # 1 or 2, based on nesting level in template

# The three closer types and their openers
CLOSER_PAIRS = [
    ("(", ")"),
    ("[", "]"),
    ("{", "}"),
]

# Minimal-pair prompt templates: 10 per closer type, varied nesting depth and context
TEMPLATES_PER_TYPE = [
    # depth-1 function call patterns (indices 0–4)
    "{o}x{c}",
    "{o}x, y{c}",
    "{o}x, y, z{c}",
    "f{o}x{c}",
    "f{o}x, y{c}",
    # depth-2 nested patterns (indices 5–9)
    "{o}{o}x{c}{c}",
    "{o}f{o}x{c}{c}",
    "g{o}{o}x{c}{c}",
    # expression patterns
    "a + {o}x + y{c}",
    "return {o}x{c}",
]

def _single_token_id(text: str) -> int | None:
    ids = model.to_tokens(text, prepend_bos=False)[0].tolist()
    return ids[0] if len(ids) == 1 else None

dataset: list[BracketPair] = []

for opener, closer in CLOSER_PAIRS:
    correct_id = _single_token_id(closer)
    if correct_id is None:
        raise RuntimeError(f"Closer '{closer}' does not map to a single token")

    # Pick an incorrect closer from the other two types
    other_closers = [c for o, c in CLOSER_PAIRS if c != closer]
    # Use the first other closer that is a single token
    incorrect_id = None
    for alt in other_closers:
        tok = _single_token_id(alt)
        if tok is not None:
            incorrect_id = tok
            break
    if incorrect_id is None:
        raise RuntimeError(f"No single-token incorrect closer found for '{closer}'")

    for tmpl_idx, tmpl in enumerate(TEMPLATES_PER_TYPE):
        depth = 1 if tmpl_idx < 5 else 2

        # Prompt is everything up to (but not including) the final closer
        filled = tmpl.format(o=opener, c=closer)
        prompt_text = filled[:-1]  # strip the correct closer — model must predict it
        prompt_ids = model.to_tokens(prompt_text, prepend_bos=True)[0].tolist()

        # matched: prompt + correct closer
        matched = prompt_ids + [correct_id]
        # mismatched: prompt + incorrect closer
        mismatched = prompt_ids + [incorrect_id]

        dataset.append(BracketPair(
            matched_tokens=matched,
            mismatched_tokens=mismatched,
            correct_closer_id=correct_id,
            incorrect_closer_id=incorrect_id,
            label=f"{opener}{closer}_{tmpl[:12].replace('{o}', opener).replace('{c}', closer)}",
            closer_type=closer,
            depth=depth,
        ))

print(f"Built {len(dataset)} prompt pairs ({len(dataset)//3} per closer type)")
print("Example prompt text:", model.to_string(torch.tensor(dataset[0].matched_tokens)))

In [ ]:
# Causal metric: logit_diff = logit(correct_closer) - logit(incorrect_closer) at the
# position *before* the closer (i.e. the model predicts from the prompt prefix).
# ablation_logit_drop = baseline_logit_diff - ablated_logit_diff

@torch.no_grad()
def mean_logit_diff(dataset: list[BracketPair],
                    ablate_layer: int = None,
                    ablate_head: int = None) -> float:
    hooks = []
    if ablate_layer is not None:
        hook_name = f"blocks.{ablate_layer}.attn.hook_result"
        def _zero(z, hook, h=ablate_head):
            z = z.clone()
            z[:, :, h, :] = 0.0
            return z
        hooks = [(hook_name, _zero)]

    total = 0.0
    for ex in dataset:
        # Feed prompt (all tokens except the final closer)
        prompt_ids = ex.matched_tokens[:-1]
        tokens = torch.tensor([prompt_ids], device=DEVICE)
        logits = model.run_with_hooks(tokens, fwd_hooks=hooks) if hooks else model(tokens)
        last_logits = logits[0, -1]  # [d_vocab]
        diff = float(last_logits[ex.correct_closer_id].item()) \
             - float(last_logits[ex.incorrect_closer_id].item())
        total += diff
    return total / len(dataset)

print("Logit-diff function defined.")

In [ ]:
import numpy as np
from tqdm.auto import tqdm

SEEDS = [0, 1, 2]

CLOSER_TYPES = [")", "]", "}"]
DEPTHS = [1, 2]

# Accumulate per-seed drops: [layer][head] -> list of floats (one per seed)
all_drops          = [[[] for _ in range(N_HEADS)] for _ in range(N_LAYERS)]
all_drops_by_closer = {c: [[[] for _ in range(N_HEADS)] for _ in range(N_LAYERS)] for c in CLOSER_TYPES}
all_drops_by_depth  = {d: [[[] for _ in range(N_HEADS)] for _ in range(N_LAYERS)] for d in DEPTHS}

# Pre-split dataset subsets (prompts are deterministic; same across seeds)
dataset_by_closer = {c: [ex for ex in dataset if ex.closer_type == c] for c in CLOSER_TYPES}
dataset_by_depth  = {d: [ex for ex in dataset if ex.depth == d] for d in DEPTHS}

for seed in SEEDS:
    print(f"\n=== Seed {seed} ===")
    # Prompts are fully deterministic; seed loop confirms stability
    baseline_diff      = mean_logit_diff(dataset)
    baseline_by_closer = {c: mean_logit_diff(dataset_by_closer[c]) for c in CLOSER_TYPES}
    baseline_by_depth  = {d: mean_logit_diff(dataset_by_depth[d])  for d in DEPTHS}
    print(f"  baseline logit diff: {baseline_diff:.4f}")

    for layer in range(N_LAYERS):
        for head in tqdm(range(N_HEADS), desc=f"  layer {layer:02d}", leave=False):
            ablated_diff = mean_logit_diff(dataset, ablate_layer=layer, ablate_head=head)
            all_drops[layer][head].append(baseline_diff - ablated_diff)

            for c in CLOSER_TYPES:
                abl_c = mean_logit_diff(dataset_by_closer[c], ablate_layer=layer, ablate_head=head)
                all_drops_by_closer[c][layer][head].append(baseline_by_closer[c] - abl_c)

            for d in DEPTHS:
                abl_d = mean_logit_diff(dataset_by_depth[d], ablate_layer=layer, ablate_head=head)
                all_drops_by_depth[d][layer][head].append(baseline_by_depth[d] - abl_d)

        print(f"  layer {layer:02d} done")

print("\nSweep complete.")

In [ ]:
import pandas as pd

rows = []
for layer in range(N_LAYERS):
    for head in range(N_HEADS):
        drops      = np.array(all_drops[layer][head])
        drops_par  = np.array(all_drops_by_closer[")"][layer][head])
        drops_brk  = np.array(all_drops_by_closer["]"][layer][head])
        drops_brc  = np.array(all_drops_by_closer["}"][layer][head])
        drops_d1   = np.array(all_drops_by_depth[1][layer][head])
        drops_d2   = np.array(all_drops_by_depth[2][layer][head])
        rows.append({
            "layer": layer,
            "head": head,
            "mean_ablation_logit_drop": float(drops.mean()),
            "std_ablation_logit_drop":  float(drops.std()),
            "mean_drop_paren":          float(drops_par.mean()),
            "mean_drop_bracket":        float(drops_brk.mean()),
            "mean_drop_brace":          float(drops_brc.mean()),
            "mean_drop_depth1":         float(drops_d1.mean()),
            "mean_drop_depth2":         float(drops_d2.mean()),
        })

results_df = (
    pd.DataFrame(rows)
    .sort_values("mean_ablation_logit_drop", ascending=False)
    .reset_index(drop=True)
)

print("Top-20 heads by mean ablation logit drop:")
results_df.head(20)

In [ ]:
from google.colab import files

csv_path = "bracket_exhaustive_results.csv"
results_df.to_csv(csv_path, index=False)
print(f"Saved {len(results_df)} rows to {csv_path}")
files.download(csv_path)